<h1>PCD — Labo 2 : Détection d'outiliers avec LOF</h1>
<h4> Auteurs : Edison Sahitaj & Adam Gruber <br>
Date : 01.03.2026 </h4>

---
<h2> Objectif </h2>
Détecter des valeurs aberrantes dans un dataset et évaluer la performance de la méthode LOF (Local Outlier Factor)

<h2 > Etape 1 : Imports & configuration </h2>

In [6]:
import numpy as np
import pandas as pd

from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

<h2> Etape 2 : Chargement et compréhension des données </h2>

<h3> ALOI (Amsterdam Library of Object Images) Dataset </h3>
Ce dataset contient des descripteurs numériques d'images d'objets. Chaque ligne correspond à une observation représentée par un vecteur de caractéristiques.

In [31]:
df = pd.read_csv(r"C:\Users\kstar\PCD\Labo2\data\ALOI.csv.gz", compression="gzip")
DATASET_NAME = "ALOI"

print("Dataset:", DATASET_NAME)
print("Shape:", df.shape)
df.head()

Dataset: ALOI
Shape: (1000, 2)


,# Data set size: 49534 data type: DoubleVector,dim=27
0,bylabel 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1....,NaN
1,KNN-001 0.0673781370351139 0.01046398024632717...,NaN
2,KNN-002 0.067556343750446 0.010866386451747896...,NaN
3,KNN-003 0.06991871960537412 0.0115219604938058...,NaN
4,KNN-004 0.07115080132862388 0.0116169508659328...,NaN


<h3> Glass Dataset </h3>
Ce dataset regroupe des mesures physico chimiques sur des échantillons de verre par exemple l'indice de réfraction, la composition en oxydes...

In [28]:
df = pd.read_csv(r"C:\Users\kstar\PCD\Labo2\data\Glass.csv.gz", compression="gzip")
DATASET_NAME = "Glass"

print("Dataset:", DATASET_NAME)
print("Shape:", df.shape)
df.head()

Dataset: Glass
Shape: (1197, 2)


,# Data set size: 214 data type: DoubleVector,dim=7
0,bylabel 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 0....,NaN
1,KNN-001 0.04185610079785267 0.0977802348176767...,NaN
2,KNN-002 0.04782060554823622 0.0996108751743503...,NaN
3,KNN-003 0.04804756729533768 0.1059049793258088...,NaN
4,KNN-004 0.04921024617902255 0.1165244771796895...,NaN


<h3> Ionosphere Dataset </h3>
Ionosphere contient des mesures radar liées à l'ionosphère. Les features décrivent des propriétés du signal, le dataset sépare des retours bon au mauvais.

In [29]:
df = pd.read_csv(r"C:\Users\kstar\PCD\Labo2\data\Ionosphere.csv.gz", compression="gzip")
DATASET_NAME = "Ionosphere"

print("Dataset:", DATASET_NAME)
print("Shape:", df.shape)
df.head()

Dataset: Ionosphere
Shape: (1197, 2)


,# Data set size: 351 data type: DoubleVector,dim=32
0,bylabel 0.0 1.0 0.0 1.0 0.0 1.0 0.0 1.0 0.0 1....,NaN
1,KNN-001 0.45201553294660135 1.1754889255007892...,NaN
2,KNN-002 0.5848638009186413 1.2702656777619394 ...,NaN
3,KNN-003 0.6025089413236953 1.3094393436887406 ...,NaN
4,KNN-004 0.6093526618059529 1.3479127486877627 ...,NaN


In [33]:
df.isna().sum().sort_values(ascending=False).head(20) # Count the number of outliers

dim=27                                            1000
# Data set size: 49534 data type: DoubleVector       0
dtype: int64

<h3> Séparer x et y </h3>

In [36]:
label_col = None

if label_col and label_col in df.columns:
    y = df[label_col].copy()
    x == df.drop(columns=[label_col]).copy()
else:
    y = None
    x = df.copy()

x.head()

,# Data set size: 49534 data type: DoubleVector,dim=27
0,bylabel 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1.0 1....,NaN
1,KNN-001 0.0673781370351139 0.01046398024632717...,NaN
2,KNN-002 0.067556343750446 0.010866386451747896...,NaN
3,KNN-003 0.06991871960537412 0.0115219604938058...,NaN
4,KNN-004 0.07115080132862388 0.0116169508659328...,NaN


<h2> Etape 3 : Prétraitement </h2>

In [44]:
x_num = x.select_dtypes(include=[np.number]).copy()

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_num)

x_num.shape

C:\Users\kstar\miniconda3\envs\pcd-labo2\Lib\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
C:\Users\kstar\miniconda3\envs\pcd-labo2\Lib\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
C:\Users\kstar\miniconda3\envs\pcd-labo2\Lib\site-packages\sklearn\utils\extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


(1000, 1)